In [ ]:
# Install Nipype and dependencies
!pip install nipype[all] nibabel scipy numpy traits pandas matplotlib seaborn nilearn

# Install other dependencies like FSL (if needed)
!apt-get install -y fsl


In [ ]:
import os
from nipype import Node, Workflow
from nipype.interfaces.io import SelectFiles, DataSink
from nipype.interfaces.utility import IdentityInterface, Function
from nipype.interfaces.fsl import BET, MCFLIRT
import numpy as np
from nilearn.input_data import NiftiLabelsMasker
from nilearn import datasets
import nilearn.plotting as plotting
import matplotlib.pyplot as plt

In [ ]:
# Define base directory
base_dir = os.path.abspath('./nipype_working_dir')

# Create a workflow
workflow = Workflow(name='connectivity_workflow')
workflow.base_dir = base_dir

# Infosource to iterate over subject IDs
infosource = Node(IdentityInterface(fields=['subject_id']), name="infosource")
infosource.iterables = [('subject_id', ['sub-10159', 'sub-10171'])]

# SelectFiles to get the data
templates = {'func': os.path.join(data_dir, 'sub-{subject_id}/func/sub-{subject_id}_task-rest_bold.nii.gz')}
selectfiles = Node(SelectFiles(templates), name="selectfiles")

# Skull stripping using FSL BET
skullstrip = Node(BET(mask=True), name="skullstrip")

# Motion correction using FSL MCFLIRT
motion_correct = Node(MCFLIRT(mean_vol=True), name="motion_correct")

# Function to extract time series and compute connectivity matrix
def extract_time_series_and_compute_connectivity(func_file):
    atlas = datasets.fetch_atlas_aal()
    masker = NiftiLabelsMasker(labels_img=atlas['maps'], standardize=True)
    time_series = masker.fit_transform(func_file)
    correlation_matrix = np.corrcoef(time_series.T)
    return correlation_matrix

# Node for extracting time series and computing connectivity matrix
connectivity = Node(Function(input_names=['func_file'],
                             output_names=['correlation_matrix'],
                             function=extract_time_series_and_compute_connectivity),
                    name="connectivity")

# DataSink to store outputs
datasink = Node(DataSink(base_directory=os.path.join(base_dir, 'output')), name="datasink")

# Connect the nodes
workflow.connect([
    (infosource, selectfiles, [('subject_id', 'subject_id')]),
    (selectfiles, skullstrip, [('func', 'in_file')]),
    (skullstrip, motion_correct, [('out_file', 'in_file')]),
    (motion_correct, connectivity, [('out_file', 'func_file')]),
    (connectivity, datasink, [('correlation_matrix', 'correlation_matrices')])
])

# Run the workflow
workflow.run('MultiProc', plugin_args={'n_procs': 4})


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Load and plot the connectivity matrix
output_dir = os.path.join(base_dir, 'output/correlation_matrices')
subject_id = 'sub-10159'  # Example subject
matrix_file = os.path.join(output_dir, f'_{subject_id}.npy')
correlation_matrix = np.load(matrix_file)

plt.figure(figsize=(10, 10))
plt.imshow(correlation_matrix, vmin=-1, vmax=1, cmap='coolwarm')
plt.colorbar()
plt.title(f'Connectivity Matrix for {subject_id}')
plt.show()